In [1]:
import os
import sys
import pandas as pd
import geopandas as gpd
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(project_root, "src"))

# import config variables
from config import DATA_RAW, DATA_PROCESSED, PLANET_API_KEY
from explore_points import process_points
from mosaics import get_mosaics_for_aoi, expand_points_by_month

In [2]:
shp_path = f"{DATA_PROCESSED}/shp/Construction_sites_WGS84.shp"
out_gpkg_path = f"{DATA_PROCESSED}/gpkg/aoi_bbox.gpkg"
out_json_path = f"{DATA_PROCESSED}/gpkg/aoi_bbox.json"

In [3]:
min_d, max_d, bbox = process_points(shp_path, out_gpkg_path, out_json_path)
min_d, max_d, bbox

Saved AOI points to D:\LGL\DATA\GIT\data\processed/gpkg/aoi_bbox.gpkg


(Timestamp('2020-06-01 00:00:00'),
 Timestamp('2025-11-30 00:00:00'),
 [8.275948994286598, 48.69081463561817, 9.315319944733018, 49.49133849492828])

In [4]:
df_mosaics = get_mosaics_for_aoi(
    bbox,
    min_d,
    max_d,
    PLANET_API_KEY,
    name_prefix="ps_",
    save_path=f"{DATA_PROCESSED}/mosaics_ref.csv"
)
df_mosaics.head(2)

,mosaic_id,name,time_start,time_end,year_month
197,2c61ff49-c7d6-471c-a042-0675d71f5d77,ps_monthly_sen2_normalized_analytic_8b_sr_subs...,2020-05-01,2020-06-01,2020-05
198,dbd62c78-d3b0-4923-a841-8a44c9ae0723,ps_monthly_sen2_normalized_analytic_8b_sr_subs...,2020-06-01,2020-07-01,2020-06


In [5]:
# points_gdf con fid_1, Start, End, Ongoing, geometry
points_gdf = gpd.read_file(shp_path)

mosaics_df = pd.read_csv(f"{DATA_PROCESSED}/mosaics_ref.csv")

points_by_month = expand_points_by_month(
    points_gdf=points_gdf,
    mosaics_df=mosaics_df,
    id_col="order",
    date_start_col="Start",
    date_end_col="End",
    ongoing_col="Ongoing",
    month_margin=0,
)

points_by_month.head(2)


,order,Start,End,Ongoing,geometry,year_month,mosaic_id
0,145,2021-03-01,2025-11-30,0.0,MULTIPOINT ((9.14698 48.69336)),2021-03,e1056ead-2018-492f-81fe-45de29f02983
1,145,2021-03-01,2025-11-30,0.0,MULTIPOINT ((9.14698 48.69336)),2021-04,1d47a0b7-d0fe-4b50-a56e-05df6346e1f4


In [6]:
from mosaics import get_quads_for_points

quads_gdf = get_quads_for_points(
    points_by_month=points_by_month,
    api_key=PLANET_API_KEY,
    buffer_deg=0.0005   # optional small buffer (~50m in lon-lat)
)

quads_gdf.head()

,mosaic_id,quad_id,geometry
0,e1056ead-2018-492f-81fe-45de29f02983,1071-1348,"POLYGON ((8.4375 49.38237, 8.4375 49.49667, 8...."
1,e1056ead-2018-492f-81fe-45de29f02983,1072-1348,"POLYGON ((8.61328 49.38237, 8.61328 49.49667, ..."
2,e1056ead-2018-492f-81fe-45de29f02983,1073-1348,"POLYGON ((8.78906 49.38237, 8.78906 49.49667, ..."
3,e1056ead-2018-492f-81fe-45de29f02983,1074-1348,"POLYGON ((8.96484 49.38237, 8.96484 49.49667, ..."
4,e1056ead-2018-492f-81fe-45de29f02983,1075-1348,"POLYGON ((9.14062 49.38237, 9.14062 49.49667, ..."


In [7]:
quads_gdf.to_file(f"{DATA_PROCESSED}/gpkg/points_quads.gpkg", driver="GPKG")

In [8]:
from mosaics import assign_quads_to_points

point_quad = assign_quads_to_points(points_by_month, quads_gdf)
point_quad.head()

,order,Start,End,Ongoing,year_month,mosaic_id,quad_id,geometry
0,145,2021-03-01,2025-11-30,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1076-1342,MULTIPOINT ((9.14698 48.69336))
1,178,2021-03-01,2022-10-31,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1076-1342,MULTIPOINT ((9.3023 48.69382))
2,191,2021-03-01,2023-11-30,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1075-1343,MULTIPOINT ((9.07058 48.8233))
3,52,2020-08-01,2022-09-01,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1072-1344,MULTIPOINT ((8.47988 48.92885))
4,159,2021-02-01,2021-05-01,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1076-1342,MULTIPOINT ((9.26926 48.73964))


In [9]:
from mosaics import extract_quads_to_download, build_quad_index
quads_to_download = extract_quads_to_download(point_quad)
quad_index = build_quad_index(point_quad)

In [ ]:
print(f"Total quads to download: {len(quads_to_download)}")
print(f"Total unique quads in index: {len(quad_index)}")


quad_index.head()

Total quads to download: 972
Total unique quads in index: 972


,mosaic_id,quad_id,n_sites,sites
0,02ece6c3-0e17-43cc-be23-b6dedeb80f5b,1071-1343,1,[245]
1,02ece6c3-0e17-43cc-be23-b6dedeb80f5b,1071-1344,5,"[242, 259, 260, 264, 265]"
2,02ece6c3-0e17-43cc-be23-b6dedeb80f5b,1072-1342,1,[269]
3,02ece6c3-0e17-43cc-be23-b6dedeb80f5b,1072-1343,2,"[85, 339]"
4,02ece6c3-0e17-43cc-be23-b6dedeb80f5b,1072-1344,2,"[52, 254]"
